<div style="background:linear-gradient(135deg,#0a2540 0%,#1a3a5c 60%,#0f3460 100%);
            padding:40px 30px;border-radius:12px;text-align:center;margin-bottom:10px;">
  <h1 style="color:#f4a261;font-size:2em;margin:0 0 8px;">
    🔬 Ciencia de Datos en Descubrimiento de Fármacos
  </h1>
  <h2 style="color:#a8dadc;font-size:1.2em;font-weight:400;margin:0 0 16px;">
    13 ·  Docking Molecular y Validación del Protocolo: Preparación · Re-docking · RMSD · ProLIF
  </h2>
  <p style="color:#cdd6f4;font-size:0.95em;max-width:640px;margin:0 auto;line-height:1.6;">
    Universidad Nacional de Colombia · Extensión UNAL 2026<br>
    <em>Semana 6 — De la estructura 3D al sitio de unión</em>
  </p>
</div>


---
## ¿Qué es el docking molecular y por qué importa?

En los notebooks anteriores construimos modelos QSAR que predicen la **actividad biológica**
a partir de descriptores 2D. Ahora damos un paso más: queremos entender **cómo y dónde**
se une una molécula a su target proteico.

El **docking molecular** simula computacionalmente la unión de un ligando pequeño
al sitio de unión de una proteína y estima la energía libre de esa interacción.

> *Una molécula puede ser "activa" en un ensayo in vitro y, sin embargo, unirse de forma
> no específica o en un sitio equivocado. El docking nos da información estructural
> que los modelos 2D no pueden capturar.*

### ¿Qué haremos en este notebook?

| # | Sección | Herramienta | Concepto clave |
|---|---------|------------|---------------|
| 1 | Instalación y contexto | pip | Pipeline completo |
| 2 | Descarga de la estructura PDB | RCSB API | Proteína co-cristalizada |
| 3 | Inspección y separación proteína/ligando | MDAnalysis + NGLview | Universo molecular |
| 4 | Preparación de la proteína | pdb2pqr | Cargas a pH fisiológico |
| 5 | Preparación del ligando | obabel + SMILES | PDBQT desde SDF |
| 6 | Definición de la caja de docking | MDAnalysis | Centro + dimensiones |
| 7 | Re-docking de validación | AutoDock Vina | ¿Reproduce la pose cristalográfica? |
| 8 | Evaluación: RMSD simétrico | spyrmsd | Umbral ≤ 2.0 Å |
| 9 | Interacciones ProLIF: cristal vs docking | ProLIF | Fingerprint proteína-ligando |
| 10 | Interpretación farmacológica | — | ¿Qué residuos son clave? |

---
> **Entrada:** Código PDB de tu proteína de interés (en este notebook usamos **7WJO** — SARS-CoV-2 Mpro)
> **Salida:** Protocolo validado + fingerprint de interacciones → listo para NB-DOCK-02

### ¿Por qué empezamos con el re-docking?

Antes de hacer docking de moléculas nuevas, **siempre** debemos verificar que nuestro
protocolo es capaz de reproducir la pose que la naturaleza encontró (la pose cristalográfica).
Si no podemos reproducirla, el protocolo no es confiable para predecir poses de moléculas nuevas.

Esta etapa de validación se llama **re-docking** y es el equivalente computacional
de hacer un control positivo en el laboratorio.


---
## 1. Instalación de librerías

Usamos un conjunto específico de herramientas, cada una con un rol distinto:

| Librería | Función en el pipeline |
|----------|----------------------|
| `vina` | Motor de docking — AutoDock Vina desde Python |
| `MDAnalysis` | Manipulación de estructuras moleculares (PDB, PQR) |
| `nglview` | Visualización 3D interactiva en el notebook |
| `pdb2pqr` | Añade hidrógenos y cargas a la proteína a pH dado |
| `openbabel-wheel` | Conversión entre formatos moleculares (SDF ↔ PDBQT) |
| `spyrmsd` | RMSD simétrico — corrige errores por simetría molecular |
| `prolif` | Fingerprints de interacción proteína-ligando |
| `rcsbsearchapi` | Búsqueda y descarga desde el PDB |


In [ ]:
# ── Instalar todas las librerías necesarias ─────────────────────────────────
# Esto puede tardar 2-3 minutos en Colab

!pip install rdkit vina nglview spyrmsd rcsbsearchapi MDAnalysis pdb2pqr openbabel-wheel prolif --quiet
!pip install git+https://github.com/openmm/pdbfixer.git --quiet

print("✅ Librerías instaladas")


In [ ]:
# ── Configuración inicial ────────────────────────────────────────────────────
import os
import requests
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# # Habilitar widgets interactivos en Colab
# from google.colab import output
# output.enable_custom_widget_manager()

import MDAnalysis as mda
import nglview as nv
from rdkit import Chem
from rdkit.Chem.AllChem import AssignBondOrdersFromTemplate

print("✅ Todo listo para comenzar")
print()
print("Pipeline que seguiremos:")
print("  PDB → separar prot/lig → pdb2pqr → PDBQT")
print("       ↓")
print("  Ligando SDF → PDBQT (obabel)")
print("       ↓")
print("  AutoDock Vina → poses PDBQT → SDF (openbabel)")
print("       ↓")
print("  RMSD con spyrmsd → validación del protocolo")
print("       ↓")
print("  ProLIF → fingerprint de interacciones")


---
## 2. Descarga de la estructura PDB

Usamos como ejemplo la estructura **7WJO**: SARS-CoV-2 Mpro (proteasa principal)
co-cristalizada con el ligando **BGI** (un peptidomimético inhibidor).

Mpro es el target de **nirmatrelvir** (Paxlovid) — el antiviral que discutimos en la semana 1.

### ¿Por qué 7WJO?
- Alta resolución (1.73 Å) — poses bien definidas
- Ligando co-cristalizado bien caracterizado
- Target farmacológicamente relevante y con datos en ChEMBL

> 💡 **Para tu proyecto:** cambia `PDB_ID` y `LIGAND_CODE` por los de tu target.
> Encuentra el código PDB en [rcsb.org](https://www.rcsb.org) buscando tu proteína.


In [ ]:
# ── Parámetros del sistema ── ¡AJUSTA AQUÍ PARA TU PROYECTO! ────────────────
PDB_ID       = '7WJO'    # ← Código PDB de tu estructura (4 caracteres)
LIGAND_CODE  = 'BGI'     # ← Código del ligando co-cristalizado (3 caracteres)
CHAIN_ID     = 'A'       # ← Cadena proteica de interés

# SMILES del ligando (búscalo en PubChem o ChEMBL)
SMILES_LIGANDO = 'C[C@H]([C@@H](c1ccc(cn1)O)O)[C@@H](C(=O)N[C@@H]([C@@H]2[C@H]([C@H]([C@@H](O2)N3C=CC(=O)NC3=O)O)O)C(=O)O)N'

# Crear carpetas de trabajo
protein_directory = "estructuras"
pdbqt_directory   = "pdbqt"
os.makedirs(protein_directory, exist_ok=True)
os.makedirs(pdbqt_directory, exist_ok=True)

print(f"Sistema de trabajo:")
print(f"  Target:  {PDB_ID} (SARS-CoV-2 Mpro)")
print(f"  Ligando: {LIGAND_CODE}")
print(f"  Cadena:  {CHAIN_ID}")
print(f"  Carpetas creadas: {protein_directory}/, {pdbqt_directory}/")


---
## 3. Inspección y separación de proteína y ligando

El archivo PDB contiene **todo** junto: proteína, ligando, agua y cofactores.
Necesitamos separar cada componente para el pipeline de docking.

Usamos **MDAnalysis** que carga el PDB como un "universo molecular" —
una representación completa de todos los átomos y sus coordenadas.


In [ ]:
# ── Cargar el PDB con MDAnalysis ─────────────────────────────────────────────
u = mda.Universe(f"{protein_directory}/{PDB_ID}.pdb")

print(f"Universo molecular cargado: {PDB_ID}")
print(f"  Átomos totales:    {len(u.atoms)}")
print(f"  Residuos totales:  {len(u.residues)}")
print(f"  Cadenas (segids):  {list(set(u.segments.segids))}")
print()

# Identificar los ligandos presentes (HETATM que no son agua)
hetatm = u.select_atoms("not protein and not resname HOH WAT")
residuos_het = list(set(hetatm.resnames))
print(f"  Moléculas no proteicas (excl. agua): {residuos_het}")


In [ ]:
# ── Visualizar la estructura completa ────────────────────────────────────────
view = nv.show_mdanalysis(u)
view.add_representation('cartoon', selection='protein', color='lightblue')
view.add_representation('licorice', selection=f'resname {LIGAND_CODE}', color='yellow')
view.center()
view


In [ ]:
# ── Seleccionar proteína y ligando por separado ──────────────────────────────
# Seleccionamos solo la cadena A para evitar ambigüedades en estructuras diméricas
protein = u.select_atoms(f"protein and segid {CHAIN_ID}")
ligand  = u.select_atoms(f"resname {LIGAND_CODE} and segid {CHAIN_ID}")

print(f"Selección proteína (cadena {CHAIN_ID}):")
print(f"  Átomos: {len(protein)}")
print(f"  Residuos: {len(protein.residues)}")
print()
print(f"Selección ligando ({LIGAND_CODE}, cadena {CHAIN_ID}):")
print(f"  Átomos: {len(ligand)}")
print(f"  Residuos: {len(ligand.residues)}")

# Si el ligando tiene 0 átomos, ajustar el chainID
if len(ligand) == 0:
    print()
    print("⚠️  El ligando no se encontró en la cadena A.")
    print("   Probando sin restricción de cadena...")
    ligand = u.select_atoms(f"resname {LIGAND_CODE}")
    print(f"   Ligando encontrado: {len(ligand)} átomos")


In [ ]:
# ── Visualización proteína + ligando en el sitio de unión ───────────────────
n_view = nv.NGLWidget()

prot_comp = n_view.add_component(protein)
prot_comp.clear_representations()
prot_comp.add_cartoon(color='lightblue')
prot_comp.add_surface(opacity=0.15, color='lightblue')  # superficie del sitio de unión

lig_comp = n_view.add_component(ligand)
lig_comp.clear_representations()
lig_comp.add_licorice(color='yellow')

n_view.center()
n_view


In [ ]:
# ── Guardar la cadena A de la proteína ──────────────────────────────────────
# Guardamos también una versión sin agua para el docking
protein_clean = u.select_atoms(f"protein and segid {CHAIN_ID}")
protein_clean.write(f"{protein_directory}/{PDB_ID}_a.pdb")

print(f"✅ Proteína guardada: {protein_directory}/{PDB_ID}_a.pdb")
print(f"   {len(protein_clean)} átomos")


In [ ]:
# ── Guardar el ligando en formato SDF (con coordenadas cristalográficas) ─────
# Las coordenadas originales del PDB son nuestra referencia para el RMSD

# Guardar el ligando en PDB primero
ligand.atoms.write(f"{protein_directory}/{LIGAND_CODE}_org.pdb")

# Convertir a SDF usando RDKit + SMILES (para asignar órdenes de enlace correctos)
pdb_lig = Chem.MolFromPDBFile(f'{protein_directory}/{LIGAND_CODE}_org.pdb',
                               removeHs=False, sanitize=False)
smiles_mol = Chem.MolFromSmiles(SMILES_LIGANDO)

try:
    mol_asignado = AssignBondOrdersFromTemplate(smiles_mol, pdb_lig)
    writer = Chem.SDWriter(f'{protein_directory}/{LIGAND_CODE}_org.sdf')
    writer.write(mol_asignado)
    writer.close()
    print(f"✅ Ligando guardado con órdenes de enlace: {protein_directory}/{LIGAND_CODE}_org.sdf")
except Exception as e:
    print(f"⚠️  Error asignando órdenes de enlace: {e}")
    print("   Guardando sin órdenes de enlace asignados...")
    writer = Chem.SDWriter(f'{protein_directory}/{LIGAND_CODE}_org.sdf')
    writer.write(pdb_lig)
    writer.close()


---
## 4. Preparación de la proteína: pdb2pqr

Los archivos PDB descargados del RCSB **no tienen hidrógenos** (no son visibles por rayos X)
y no tienen cargas asignadas. AutoDock Vina necesita ambas cosas.

**pdb2pqr** resuelve este problema:
1. Añade hidrógenos a pH fisiológico (7.4)
2. Asigna el estado de protonación correcto a histidinas, ácidos y bases
3. Asigna cargas parciales con el campo de fuerzas AMBER
4. Genera un archivo PQR (PDB con cargas y radios de van der Waals)

> 💡 El pH 7.4 es el pH fisiológico — el entorno real donde actúan los fármacos.
> Cambiar el pH afecta el estado de protonación y puede cambiar significativamente
> las interacciones predichas.


In [ ]:
# ── Agregar hidrógenos y cargas con pdb2pqr ─────────────────────────────────
# El "!" ejecuta comandos de terminal desde Jupyter/Colab
# Parámetros:
#   --pdb-output   → archivo PDB con hidrógenos (para visualización)
#   --pH=7.4       → pH fisiológico
#   entrada        → PDB sin hidrógenos
#   salida.pqr     → archivo con cargas y radios (para Vina)
#   --whitespace   → formato más legible

print("Ejecutando pdb2pqr...")
print(f"  Entrada: {protein_directory}/{PDB_ID}_a.pdb")
print(f"  Salida PDB: {protein_directory}/protein_h.pdb")
print(f"  Salida PQR: {protein_directory}/{PDB_ID}_a.pqr")
print()


In [ ]:
!pdb2pqr \
    --pdb-output={protein_directory}/protein_h.pdb \
    --pH=7.4 \
    --whitespace \
    {protein_directory}/{PDB_ID}_a.pdb \
    {protein_directory}/{PDB_ID}_a.pqr

# Verificar que se creó el archivo
if os.path.exists(f"{protein_directory}/{PDB_ID}_a.pqr"):
    tam = os.path.getsize(f"{protein_directory}/{PDB_ID}_a.pqr") / 1024
    print(f"\n✅ Archivo PQR generado ({tam:.0f} KB)")
else:
    print("\n❌ Error: el archivo PQR no se creó. Revisa el PDB de entrada.")


In [ ]:
# ── Convertir PQR → PDBQT (formato de Vina) ─────────────────────────────────
# MDAnalysis puede leer el PQR y escribir un PDBQT compatible con Vina

u_pqr = mda.Universe(f"{protein_directory}/{PDB_ID}_a.pqr")

# Eliminar moléculas de agua (no las necesitamos para el docking)
proteina_sin_agua = u_pqr.select_atoms("not water")

# Guardar en formato PDBQT
proteina_sin_agua.atoms.write(f"{pdbqt_directory}/{PDB_ID}.pdbqt")

print(f"✅ Proteína preparada: {pdbqt_directory}/{PDB_ID}.pdbqt")
print(f"   Átomos (sin agua): {len(proteina_sin_agua)}")


In [ ]:
# ── Corrección del encabezado PDBQT ─────────────────────────────────────────
# MDAnalysis escribe cabeceras 'TITLE' y 'CRYST1' que Vina no reconoce.
# Las reemplazamos por 'REMARK' para evitar errores.

with open(f"{pdbqt_directory}/{PDB_ID}.pdbqt", 'r') as f:
    contenido = f.read()

# Reemplazar líneas problemáticas
contenido = contenido.replace('TITLE', 'REMARK').replace('CRYST1', 'REMARK')

with open(f"{pdbqt_directory}/{PDB_ID}.pdbqt", 'w') as f:
    f.write(contenido)

# Verificar
n_lineas = contenido.count(' ')
print(f"✅ Encabezado corregido")
print(f"   Líneas en el PDBQT: {n_lineas}")
print()
print("💡 El formato PDBQT es específico de AutoDock Vina.")
print("   Cada átomo tiene dos columnas extra: carga parcial y tipo de átomo AD4.")


---
## 5. Preparación del ligando

Para el re-docking necesitamos el ligando co-cristalizado en formato PDBQT.
El PDB proporciona archivos SDF de sus ligandos — los usaremos como punto de partida.

> **¿Por qué no usamos directamente el PDB del ligando?**
> Porque los ligandos en el PDB a menudo tienen órdenes de enlace mal asignados.
> Descargamos el SDF "ideal" del PDB que sí tiene la geometría y órdenes correctos.


In [ ]:
# ── Descargar el SDF del ligando desde el PDB ───────────────────────────────
import urllib.request

url_sdf = f"https://files.rcsb.org/ligands/view/{LIGAND_CODE}_ideal.sdf"
ruta_sdf = f"{protein_directory}/{LIGAND_CODE}.sdf"

try:
    urllib.request.urlretrieve(url_sdf, ruta_sdf)

    if os.path.exists(ruta_sdf) and os.path.getsize(ruta_sdf) > 0:
        tam = os.path.getsize(ruta_sdf)
        print(f"✅ SDF del ligando descargado: {ruta_sdf} ({tam} bytes)")
        print()
        # Verificar con RDKit
        mol_test = Chem.SDMolSupplier(ruta_sdf)[0]
        if mol_test:
            print(f"   Átomos pesados: {mol_test.GetNumHeavyAtoms()}")
            print(f"   Fórmula:        {Chem.rdMolDescriptors.CalcMolFormula(mol_test)}")
    else:
        print(f"⚠️  El SDF está vacío — el ligando puede no estar en el PDB")
        print(f"   Alternativa: usa obabel para generar el SDF desde el SMILES")

except Exception as e:
    print(f"⚠️  No se pudo descargar el SDF ideal: {e}")
    print(f"   Generando desde SMILES con RDKit...")

    # Fallback: generar desde SMILES con conformero 3D
    from rdkit.Chem import AllChem
    mol = Chem.MolFromSmiles(SMILES_LIGANDO)
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
    AllChem.MMFFOptimizeMolecule(mol)
    writer = Chem.SDWriter(ruta_sdf)
    writer.write(mol)
    writer.close()
    print(f"   ✅ SDF generado desde SMILES")


In [ ]:
# ── Convertir SDF → PDBQT con openbabel ─────────────────────────────────────
# openbabel añade cargas Gasteiger al ligando (necesarias para Vina)
# El flag -p asegura que se añaden los hidrógenos con los estados de protonación correctos

print("Convirtiendo SDF → PDBQT...")
!obabel -i sdf {protein_directory}/{LIGAND_CODE}.sdf -o pdbqt -O {pdbqt_directory}/{LIGAND_CODE}.pdbqt -p

if os.path.exists(f"{pdbqt_directory}/{LIGAND_CODE}.pdbqt"):
    tam = os.path.getsize(f"{pdbqt_directory}/{LIGAND_CODE}.pdbqt")
    print(f"\n✅ Ligando preparado: {pdbqt_directory}/{LIGAND_CODE}.pdbqt ({tam} bytes)")

    # Ver las primeras líneas del PDBQT
    with open(f"{pdbqt_directory}/{LIGAND_CODE}.pdbqt") as f:
        primeras = f.readlines()[:6]
    print()
    print("Primeras líneas del PDBQT:")
    for l in primeras:
        print(f"  {l.rstrip()}")
    print("  (las dos últimas columnas son: carga parcial | tipo AD4)")
else:
    print("\n❌ Error al generar el PDBQT del ligando")


---
## 6. Definición de la caja de docking

AutoDock Vina necesita saber **dónde buscar** — no busca en toda la proteína
sino en una caja rectangular centrada en el sitio de unión.

Definimos la caja a partir del **ligando co-cristalizado**:
- **Centro:** centroide geométrico del ligando
- **Tamaño:** extensión máxima del ligando + 8 Å de margen en cada lado

> **¿Por qué 8 Å de margen?**
> Para dar espacio a que Vina explore conformaciones fuera del volumen exacto del cristal,
> pero sin explorar zonas irrelevantes de la proteína.


In [ ]:
# ── Calcular el centro de la caja ────────────────────────────────────────────
# center_of_geometry() calcula el promedio de las coordenadas XYZ de todos los átomos

pocket_center = ligand.center_of_geometry()
print("CENTRO DEL SITIO DE UNIÓN (centroide del ligando)")
print("=" * 45)
print(f"  X: {pocket_center[0]:.3f} Å")
print(f"  Y: {pocket_center[1]:.3f} Å")
print(f"  Z: {pocket_center[2]:.3f} Å")


In [ ]:
# ── Calcular el tamaño de la caja ────────────────────────────────────────────
# Tomamos la diferencia entre las posiciones máximas y mínimas del ligando
# y añadimos 8 Å de margen por cada lado

MARGEN_A = 8.0  # Ångströms de margen — puedes ajustar si el sitio es muy grande

ligand_box = ligand.positions.max(axis=0) - ligand.positions.min(axis=0) + MARGEN_A

print("TAMAÑO DE LA CAJA DE DOCKING")
print("=" * 45)
print(f"  Ancho  (X): {ligand_box[0]:.2f} Å")
print(f"  Alto   (Y): {ligand_box[1]:.2f} Å")
print(f"  Profundo(Z):{ligand_box[2]:.2f} Å")
print()
print(f"  Volumen de búsqueda: {ligand_box.prod():.0f} Å³")
print()

# Verificar que la caja no sea demasiado pequeña
if any(ligand_box < 15):
    print("⚠️  Alguna dimensión de la caja es menor a 15 Å.")
    print("   Considera aumentar el margen si el ligando es grande.")
else:
    print("✅ Dimensiones razonables para el docking.")
    print()
print("💡 Regla general: la caja debe ser ≥ 22.5 Å³ para moléculas drug-like.")


---
## 7. Re-docking con AutoDock Vina

Ahora ejecutamos el docking del ligando **que ya sabemos** cómo se une (el co-cristalizado)
para verificar que nuestro protocolo es capaz de encontrar la pose correcta.

### Función de scoring de Vina

AutoDock Vina usa una función de scoring empírica que estima la energía de unión:

$$\Delta G_{binding} \approx \Delta G_{gauss} + \Delta G_{repulsion} + \Delta G_{hbond} + \Delta G_{hydrophobic} + \Delta G_{torsion}$$

- Valores **más negativos** = unión más fuerte (energéticamente más favorable)
- Unidades: **kcal/mol**
- Rango típico para drug-like: −4 a −12 kcal/mol

> ⚠️ **Limitación importante:** el score de Vina es una aproximación. No es un predictor
> perfecto de afinidad experimental. Dos moléculas con el mismo score pueden tener
> muy distintos patrones de interacción — por eso en NB-DOCK-02 añadimos ProLIF.


In [ ]:
# ── Ejecutar el re-docking con AutoDock Vina ─────────────────────────────────
from vina import Vina

print(f"Iniciando re-docking de {LIGAND_CODE} en {PDB_ID}...")
print(f"  Exhaustividad: 16 (más alto = más exhaustivo pero más lento)")
print(f"  Poses a generar: 5")
print()

v = Vina(sf_name='vina')  # sf_name: función de scoring ('vina', 'vinardo', 'ad4')

# Cargar receptor
v.set_receptor(f"{pdbqt_directory}/{PDB_ID}.pdbqt")

# Cargar ligando
v.set_ligand_from_file(f"{pdbqt_directory}/{LIGAND_CODE}.pdbqt")

# Definir la caja de búsqueda
v.compute_vina_maps(
    center=pocket_center.tolist(),
    box_size=ligand_box.tolist()
)

# Ejecutar el docking
v.dock(exhaustiveness=16, n_poses=5)

# Guardar las poses
v.write_poses(
    f"{pdbqt_directory}/redocking.pdbqt",
    n_poses=5,
    overwrite=True
)

print("✅ Re-docking completado")


In [ ]:
# ── Ver las energías de las poses generadas ──────────────────────────────────
# Columnas: [Score total, inter, intra, torsiones, intra mejor pose]

energias = v.energies()
column_names = ["Score (kcal/mol)", "Inter", "Intra", "Torsiones", "Intra mejor pose"]
df_energias = pd.DataFrame(energias, columns=column_names)
df_energias.index = [f"Pose {i+1}" for i in range(len(df_energias))]

print("ENERGÍAS DE LAS POSES (kcal/mol)")
print("=" * 60)
print(df_energias.round(3).to_string())
print()
print("💡 Score total = Inter + Intra (energía de interacción + energía interna)")
print("   'Inter' = interacción proteína-ligando (lo más importante)")
print("   Más negativo = unión predicha más fuerte")


In [ ]:
import re
from openbabel import openbabel
from rdkit import Chem                  # <-- CORRECCIÓN: Importación base de Chem añadida
from rdkit.Chem import AllChem

def pdbqt_to_sdf(pdbqt_file_path, smiles, output_sdf_path):
    """
    Convierte un archivo PDBQT multi-modelo de Vina a un archivo SDF multi-molécula.
    Asigna órdenes de enlace desde un SMILES, reconstruye hidrógenos tridimensionales
    correctos y guarda los scores como propiedades SDF.
    """
    # Leer todos los modelos del PDBQT con openbabel
    obConversion = openbabel.OBConversion()
    obConversion.SetInAndOutFormats("pdbqt", "sdf")

    mols = []
    scores = []
    mol = openbabel.OBMol()

    with open(pdbqt_file_path) as f:
        contenido = f.read()

    # Extraer scores de los comentarios de Vina
    for linea in contenido.splitlines():
        if 'VINA RESULT' in linea:
            partes = linea.split()
            if len(partes) >= 4:
                scores.append(float(partes[3]))

    # Parsear cada modelo con OpenBabel
    obConversion.ReadString(mol, contenido)
    mols.append(openbabel.OBMol(mol))
    while obConversion.Read(mol):
        mols.append(openbabel.OBMol(mol))

    if not mols:
        print("⚠️ No se encontraron poses en el PDBQT")
        return

    # Preparar el SMILES de referencia de RDKit (Molécula plantilla)
    smiles_mol = Chem.MolFromSmiles(smiles)
    if smiles_mol is None:
        print("❌ El SMILES proporcionado no es válido para RDKit.")
        return
    
    # Asegurarnos de que la plantilla no tenga hidrógenos explícitos que estorben el macheo
    smiles_mol = Chem.RemoveHs(smiles_mol)

    escritor = Chem.SDWriter(output_sdf_path)
    poses_guardadas = 0

    for i, ob_mol in enumerate(mols):
        # Convertir la pose cruda de OpenBabel a un bloque SDF de texto
        sdf_temp = obConversion.WriteString(ob_mol)
        
        # Leer en RDKit de forma permisiva (sin sanitizar para que no explote aquí)
        rdkit_mol = Chem.MolFromMolBlock(sdf_temp, removeHs=False, sanitize=False)

        if rdkit_mol is None:
            print(f"⚠️ No se pudo parsear la pose {i+1} desde el bloque de OpenBabel. Saltando...")
            continue

        try:
            # 1. Eliminar los hidrógenos conflictivos y mal calculados por OpenBabel
            rdkit_mol_clean = Chem.RemoveHs(rdkit_mol, sanitize=False)
            
            # 2. Asignar los órdenes de enlace correctos usando el SMILES estructural de plantilla
            rdkit_mol_bo = AllChem.AssignBondOrdersFromTemplate(smiles_mol, rdkit_mol_clean)
            
            # 3. Volver a añadir los hidrógenos en 3D basándonos en las valencias reales recién asignadas
            rdkit_mol_bo = Chem.AddHs(rdkit_mol_bo, addCoords=True)
            
            # 4. Forzar la sanitización estricta de RDKit para asegurar la salud de la molécula
            Chem.SanitizeMol(rdkit_mol_bo)
            
        except Exception as e:
            # Si el algoritmo de asignación falla o la molécula sigue siendo inválida, no la guardamos
            print(f"⚠️ Pose {i+1} descartada: Falló la reconstrucción de enlaces/valencias ({e})")
            continue

        # Añadir el score de Vina como propiedad del SDF si pasa el control de calidad
        score = scores[i] if i < len(scores) else 0.0
        rdkit_mol_bo.SetDoubleProp('vina_score', score)
        rdkit_mol_bo.SetIntProp('pose_id', i + 1)
        
        # Escribir la pose totalmente limpia al archivo final
        escritor.write(rdkit_mol_bo)
        poses_guardadas += 1

    escritor.close()
    print(f"\n✅ Proceso terminado de forma segura.")
    print(f"   Poses totales en PDBQT: {len(mols)}")
    print(f"   Poses químicamente curadas y guardadas en SDF: {poses_guardadas}")

# Ejecutar la conversión
pdbqt_to_sdf(f"{pdbqt_directory}/redocking.pdbqt", SMILES_LIGANDO, f"{pdbqt_directory}/redocking.sdf")

---
## 8. Validación: RMSD simétrico con spyrmsd

El **RMSD** (Root Mean Square Deviation) mide la diferencia entre la pose predicha
por Vina y la pose cristalográfica:

$$\text{RMSD} = \sqrt{\frac{1}{N}\sum_{i=1}^{N}|r_i^{\text{docking}} - r_i^{\text{cristal}}|^2}$$

### ¿Por qué RMSD **simétrico**?

El RMSD estándar es sensible a la numeración de los átomos. En moléculas con simetría
(como anillos o grupos equivalentes), el RMSD puede ser artificialmente alto
aunque la pose visual sea correcta. **spyrmsd** corrige esto evaluando todas
las permutaciones simétricas posibles.

### Criterio de validación

| RMSD | Interpretación |
|------|---------------|
| ≤ 1.0 Å | Excelente — pose muy precisa |
| ≤ 2.0 Å | **Aceptable** — criterio estándar en la literatura |
| > 2.0 Å | El protocolo **no es válido** para este sistema |

> ⚠️ Si RMSD > 2.0 Å: revisar la preparación de la proteína, el ligando o la caja de docking.


In [ ]:
# ── Calcular RMSD simétrico entre poses de docking y la referencia cristalográfica ──
from spyrmsd import io, rmsd

# Cargar la molécula de referencia (pose cristalográfica)
ref = io.loadmol(f'{protein_directory}/{LIGAND_CODE}_org.sdf')
ref.strip()  # eliminar hidrógenos para el cálculo

# Cargar las poses del docking
poses = io.loadallmols(f'{pdbqt_directory}/redocking.sdf')

# ✅ EL FIX: Eliminar los hidrógenos de todas las poses para que coincidan con la referencia
for pose in poses:
    pose.strip()

print(f"Referencia cristalográfica: {LIGAND_CODE}_org.sdf")
print(f"Poses de docking:           {len(poses)} poses")
print()

# Extraer coordenadas y matrices de adyacencia
coords_ref = ref.coordinates
anum_ref   = ref.atomicnums
adj_ref    = ref.adjacency_matrix

coords = [mol.coordinates for mol in poses]
anum   = poses[0].atomicnums
adj    = poses[0].adjacency_matrix

# Calcular RMSD simétrico para cada pose
RMSD = rmsd.symmrmsd(
    coords_ref,
    coords,
    anum_ref,
    anum,
    adj_ref,
    adj,
    minimize=True    # minimizar el RMSD por superposición (traslación + rotación)
)

print("RMSD POR POSE vs CRISTALOGRAFÍA")
print("=" * 45)
for i, (r, score) in enumerate(zip(RMSD, df_energias["Score (kcal/mol)"])):
    estado = "✅" if r <= 2.0 else "❌"
    barra = '█' * int(min(r, 5) * 4) if r > 0 else '·'
    print(f"  Pose {i+1}: RMSD = {r:.3f} Å  Score = {score:.2f} kcal/mol  {estado}  {barra}")

mejor_rmsd = min(RMSD)
mejor_pose = RMSD.index(mejor_rmsd) + 1
print()
print(f"  Mejor RMSD: {mejor_rmsd:.3f} Å (Pose {mejor_pose})")

if mejor_rmsd <= 2.0:
    print(f"  ✅ PROTOCOLO VÁLIDO — el docking reproduce la pose cristalográfica")
    print(f"     Podemos proceder al docking de moléculas nuevas (NB-DOCK-02)")
else:
    print(f"  ❌ PROTOCOLO NO VÁLIDO — revisar preparación antes de continuar")
    print(f"     Sugerencias:")
    print(f"     - Verificar que el ligando correcto fue seleccionado ({LIGAND_CODE})")
    print(f"     - Aumentar la exhaustividad de Vina (exhaustiveness=32)")
    print(f"     - Revisar el estado de protonación con pdb2pqr")

In [ ]:
# ── Guardar tabla de resultados ──────────────────────────────────────────────
df_energias['RMSD (Å)'] = RMSD
df_energias['Válida (≤2Å)'] = ['✅' if r <= 2.0 else '❌' for r in RMSD]

print("TABLA DE RESULTADOS DEL RE-DOCKING")
print("=" * 65)
print(df_energias.round(3).to_string())

# Guardar CSV
df_energias.to_csv(f'{protein_directory}/redocking_results.csv')
print(f"\n✅ Resultados guardados: {protein_directory}/redocking_results.csv")


---
## 9. Fingerprints de interacción proteína-ligando con ProLIF

El score de Vina nos dice **cuánto** se une la molécula, pero no **cómo**.
**ProLIF** (Protein-Ligand Interaction Fingerprints) extrae los contactos
específicos entre el ligando y los residuos de la proteína.

### Tipos de interacción que detecta ProLIF

| Interacción | Descripción | Energía típica |
|-------------|-------------|---------------|
| **HBDonor** | El ligando dona un H a un aceptor de la proteína | −1 a −5 kcal/mol |
| **HBAcceptor** | El ligando acepta un H del donador de la proteína | −1 a −5 kcal/mol |
| **Hydrophobic** | Contacto hidrofóbico (C···C) | −0.5 a −2 kcal/mol |
| **PiStacking** | Apilamiento π-π entre anillos aromáticos | −1 a −3 kcal/mol |
| **PiCation** | Interacción π-catión | −1 a −4 kcal/mol |
| **VdWContact** | Contacto de van der Waals | −0.1 a −0.5 kcal/mol |
| **Cationic/Anionic** | Interacción iónica | −3 a −10 kcal/mol |

Comparando el fingerprint del **cristal** con el del **docking**, podemos verificar
que el modelo reproduce las interacciones farmacológicamente relevantes.


In [ ]:
# ── Cargar la proteína preparada para ProLIF ─────────────────────────────────
import prolif as plf

rdkit_prot = Chem.MolFromPDBFile(
    f'{protein_directory}/{PDB_ID}_a.pdb',
    removeHs=False,
    sanitize=False
)

protein_mol = plf.Molecule(rdkit_prot)
print(f"✅ Proteína cargada para ProLIF")
print(f"   Residuos: {len(protein_mol.GetAtoms())} átomos")


In [ ]:
# ── Fingerprint del ligando CRISTALOGRÁFICO ──────────────────────────────────
# Esta es nuestra referencia — las interacciones "correctas" que queremos reproducir

ligand_mol_cristal = plf.sdf_supplier(f'{protein_directory}/{LIGAND_CODE}_org.sdf')[0]

fp_cristal = plf.Fingerprint(vicinity_cutoff=8.0, count=True)
fp_cristal.run_from_iterable([ligand_mol_cristal], protein_mol)

df_cristal = fp_cristal.to_dataframe()

print("INTERACCIONES DEL LIGANDO CRISTALOGRÁFICO")
print("=" * 55)
print(f"  Residuos con contacto: {df_cristal.shape[1]}")
print()
# Mostrar solo los residuos con interacciones
df_t = df_cristal.T
df_t_positivos = df_t[df_t.any(axis=1)]
print(df_t_positivos.to_string())


In [ ]:
# ── Visualización de la red de interacciones (cristal) ──────────────────────
print("Red de interacciones — Ligando cristalográfico")
fp_cristal.plot_lignetwork(ligand_mol_cristal)


In [ ]:
# ── Fingerprint de las poses del DOCKING ────────────────────────────────────
poses_docking = plf.sdf_supplier(f'{pdbqt_directory}/redocking.sdf')

fp_docking = plf.Fingerprint(vicinity_cutoff=8.0, count=True)
fp_docking.run_from_iterable(poses_docking, protein_mol)

df_docking = fp_docking.to_dataframe()

print("INTERACCIONES DE LAS POSES DE DOCKING")
print("=" * 55)
print(f"  Poses analizadas: {len(poses_docking)}")
df_dock_t = df_docking.T
df_dock_positivos = df_dock_t[df_dock_t.any(axis=1)]
print(df_dock_positivos.to_string())


In [ ]:
# ── Visualización de la red de interacciones (mejor pose docking) ────────────
# Tomamos la pose con mejor RMSD (más parecida al cristal)
idx_mejor = RMSD.index(min(RMSD))
print(f"Visualizando interacciones de la Pose {idx_mejor+1} (mejor RMSD = {min(RMSD):.3f} Å)")
fp_docking.plot_lignetwork(poses_docking[idx_mejor])


---
## 10. Interpretación farmacológica

Ahora comparamos las interacciones del cristal con las del docking para evaluar
la calidad del protocolo desde el punto de vista farmacológico.


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# ✅ EL FIX MATEMÁTICO: Alinear las columnas de ambos DataFrames 
# Esto crea un espacio común de interacciones y rellena con 0 donde no hubo contacto
df_cristal_aligned, df_docking_aligned = df_cristal.align(df_docking, join='outer', axis=1, fill_value=0)

# Convertir a vectores binarios idénticos en tamaño
vec_cristal = df_cristal_aligned.values.astype(float)
vec_poses   = df_docking_aligned.values.astype(float)

print("COMPARACIÓN CRISTAL vs POSES DE DOCKING (CORREGIDO)")
print("(Similitud coseno del fingerprint de interacciones)")
print("=" * 55)

for i in range(min(len(vec_poses), 5)):
    vec_pose_i = vec_poses[i:i+1]
    
    # Ahora sí podemos calcular la similitud real
    sim = cosine_similarity(vec_cristal, vec_pose_i)[0][0]
    
    rmsd_i = RMSD[i] if i < len(RMSD) else 0.0
    score_i = df_energias["Score (kcal/mol)"].iloc[i]
    
    estado = "✅" if sim > 0.5 and rmsd_i <= 2.0 else "❌"
    print(f"  Pose {i+1}: similitud = {sim:.3f}   RMSD = {rmsd_i:.3f} Å   Score = {score_i:.2f}   {estado}")


In [ ]:
# ── Identificar los residuos clave del sitio de unión ───────────────────────
# Los residuos que interactúan con el ligando cristalográfico son los más importantes
# para el diseño de nuevos inhibidores

print("RESIDUOS CLAVE DEL SITIO DE UNIÓN")
print(f"Target: {PDB_ID} — {LIGAND_CODE}")
print("=" * 55)
print()

# Extraer nombres de residuos de las columnas del DataFrame
if len(df_cristal.columns) > 0:
    residuos_columnas = df_cristal.columns

    # Agrupar interacciones por residuo
    interacciones_por_residuo = {}
    for col in residuos_columnas:
        if isinstance(col, tuple):
            residuo = col[0] if len(col) > 0 else str(col)
            tipo_int = col[1] if len(col) > 1 else ''
        else:
            partes = str(col).split('.')
            residuo = partes[0] if partes else str(col)
            tipo_int = partes[1] if len(partes) > 1 else ''

        if df_cristal[col].any():
            if residuo not in interacciones_por_residuo:
                interacciones_por_residuo[residuo] = []
            interacciones_por_residuo[residuo].append(tipo_int)

    for residuo, tipos in sorted(interacciones_por_residuo.items()):
        tipos_str = ', '.join(set(tipos))
        print(f"  {residuo:<15} → {tipos_str}")

print()
print("💡 Estos residuos son el 'farmacóforo' del sitio de unión.")
print("   En drug design, los nuevos compuestos deben preservar")
print("   las interacciones con los residuos más críticos.")
print()
print(f"→ Estos datos se usarán en NB-DOCK-02 para el scoring compuesto:")
print(f"  Score final = (Score Vina normalizado + Similitud coseno ProLIF) / 2")


---
## ✅ Resumen del notebook

| Paso | Herramienta | Resultado |
|------|-------------|-----------|
| **Descarga PDB** | RCSB API + `requests` | Estructura 3D del target |
| **Separación prot/lig** | `MDAnalysis` | Componentes listos por separado |
| **Cargas y H** | `pdb2pqr` | Proteína a pH 7.4 con cargas AMBER |
| **PDBQT proteína** | `MDAnalysis` | Formato compatible con Vina |
| **PDBQT ligando** | `obabel` | SDF → PDBQT con cargas Gasteiger |
| **Caja de docking** | `MDAnalysis` | Centro + dimensiones desde el ligando |
| **Re-docking** | `AutoDock Vina` | 5 poses + energías (kcal/mol) |
| **Validación RMSD** | `spyrmsd` | ¿Pose ≤ 2.0 Å del cristal? |
| **Fingerprint cristal** | `ProLIF` | Interacciones de referencia |
| **Fingerprint docking** | `ProLIF` | Interacciones de las poses |
| **Comparación** | Similitud coseno | ¿El docking reproduce las interacciones? |

## ✅ Criterio de validación del protocolo

Un protocolo de docking se considera **válido** si:
1. RMSD ≤ 2.0 Å para la mejor pose ← criterio estructural
2. Similitud de fingerprint ProLIF > 0.5 ← criterio farmacológico

Si ambos criterios se cumplen, el protocolo puede usarse en **NB-DOCK-02**
para hacer docking en batch de las moléculas activas de ChEMBL.

## 📅 Siguiente notebook: NB-DOCK-02

Con el protocolo validado, en NB-DOCK-02 haremos:
- Docking en batch de las moléculas activas de ChEMBL (del NB-DATA-02)
- Fingerprints ProLIF para todas las poses
- **Scoring compuesto** = (Score Vina normalizado + Similitud coseno ProLIF) / 2
- Ranking final y selección de candidatos

---
*NB-DOCK-01 · Ciencia de Datos en Descubrimiento de Fármacos · UNAL 2026*  
*Protocolo basado en: AutoDock Vina, MDAnalysis, ProLIF, spyrmsd*  
*Estructura ejemplo: 7WJO (SARS-CoV-2 Mpro) — target de nirmatrelvir/Paxlovid*
